# MIRepNet x Reference-Mismatch on BCIC-IV-2a

Same experiment set and same reporting as `reve-reference-mismatch.ipynb`, but the
input pipeline is **MIRepNet's own**, not REVE's. Each model is run the way its
authors intended; a cross-model comparison is only meaningful once both are
faithful, so the two notebooks share structure and metrics but not preprocessing.

MIRepNet's released pipeline (paper Sec. III-B, code `utils/utils.py`):
8-30 Hz band, 250 Hz, cue-locked 2-6 s, Euclidean Alignment, then inverse-distance
interpolation onto a fixed 45-electrode channel template.

The anchor differs too. REVE hardcodes CAR, so `ANCHOR = "car"` there. MIRepNet
re-references nothing and consumes whatever the dataset ships, so the faithful
anchor here is `native`.

## 0. Environment and MIRepNet encoder

In [ ]:
import os, sys, gc, types, pathlib, subprocess, importlib, warnings

WORK         = pathlib.Path("/kaggle/working")
REFSHIFT_DIR = WORK / "Reference-Mismatch-MI-Net"
MIREP_DIR    = WORK / "MIRepNet"
os.chdir(WORK)

for path, url in [(REFSHIFT_DIR, "https://github.com/JatinArutla/Reference-Mismatch-MI-Net"),
                  (MIREP_DIR,    "https://github.com/staraink/MIRepNet")]:
    if path.exists():
        subprocess.run(["rm", "-rf", str(path)], check=True)
    subprocess.run(["git", "clone", "--depth", "1", url, str(path)], check=True)

PIP = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"]
subprocess.run(PIP + ["moabb==1.5.0", "mne==1.11.0", "mne-bids>=0.18",
                      "pyriemann>=0.5", "einops>=0.8"], check=True)
subprocess.run(PIP + ["-e", str(REFSHIFT_DIR), "--no-deps"], check=True)

# `pip install -e` writes a .pth that `site` reads only at interpreter startup, so
# a running kernel never sees it. Put the repo roots on sys.path instead.
sys.path.insert(0, str(REFSHIFT_DIR))
sys.path.insert(0, str(MIREP_DIR))
importlib.invalidate_caches()
sys.modules.setdefault("wandb", types.ModuleType("wandb"))   # imported by model/mlm.py, unused

import numpy as np, pandas as pd, torch, mne, refshift, moabb
from model.mlm import mlm_mask
mne.set_log_level("ERROR"); warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "Enable a Kaggle GPU accelerator."
print("refshift:", refshift.__file__, "| moabb:", moabb.__version__, "| device:", DEVICE)

### 0b. Kaggle data wiring and weights

In [ ]:
# Point MOABB at the attached Kaggle copy of IV-2a. This must run before any
# get_data() call: without it MNE_DATA is unset, MOABB falls back to /root/mne_data,
# and every subject is re-downloaded into a path that is not persisted.
from refshift import setup_kaggle_env

DEFAULT_IV2A_ROOT = pathlib.Path(
    "/kaggle/input/datasets/delhialli/four-class-motor-imagery-bnci-001-2014")
if DEFAULT_IV2A_ROOT.exists():
    iv2a_source = DEFAULT_IV2A_ROOT
else:                                   # slugs change; find it, but never download
    cands = sorted(pathlib.Path("/kaggle/input").rglob("A01T.mat"))
    roots = list(dict.fromkeys(c.parent for c in cands if (c.parent/"A01E.mat").exists()))
    if len(roots) != 1:
        raise FileNotFoundError("Set REFSHIFT_IV2A_ROOT explicitly; found: " + str(roots))
    iv2a_source = roots[0]

os.environ["REFSHIFT_IV2A_ROOT"] = str(iv2a_source)
setup_kaggle_env(mne_data=str(WORK/"mne_data"), moabb_results=str(WORK/"moabb_results"),
                 symlink_datasets=["iv2a"], thread_cap=1, verbose=True)

CACHE = WORK / "mne_data" / "MNE-bnci-data" / "~bci" / "database" / "001-2014"
missing = [f for f in ("A01T.mat", "A01E.mat")
           if not ((CACHE/f).exists() or (CACHE/f).is_symlink())]
if missing:
    raise FileNotFoundError(f"IV-2a symlinks not created ({missing}); "
                            f"refusing to let MOABB download. Source: {iv2a_source}")
print("IV-2a source:", iv2a_source, "\nMOABB cache :", CACHE)

# MIRepNet ships its weights in the repo; fall back to the Hub only if absent.
WEIGHTS = MIREP_DIR / "weight" / "MIRepNet.pth"
if not WEIGHTS.exists() or WEIGHTS.stat().st_size < 1_000_000:
    from huggingface_hub import hf_hub_download
    subprocess.run(["cp", hf_hub_download(repo_id="starself/MIRepNet",
                                          filename="MIRepNet.pth"), str(WEIGHTS)], check=True)
print(f"weights     : {WEIGHTS.stat().st_size/1e6:.1f} MB")

In [ ]:
def load_encoder():
    m = mlm_mask(emb_size=256, depth=6, n_classes=3,
                 pretrainmode=False, pretrain=str(WEIGHTS)).to(DEVICE)
    # init_from_pretrained filters to matching keys before load_state_dict, so its
    # strict=True never fires. Verify the encoder actually loaded.
    ck = torch.load(WEIGHTS, map_location="cpu")
    enc = [k for k in m.state_dict() if k.startswith(("embedding.", "transformer."))]
    bad = [k for k in enc if k not in ck or ck[k].shape != m.state_dict()[k].shape]
    assert not bad, f"encoder tensors not loaded: {bad[:5]}"
    print(f"encoder loaded: {len(enc)} tensors, 0 missing, {sum(p.numel() for p in m.parameters())/1e6:.1f}M params")
    return m.eval()

mirep = load_encoder()

## 1. Configuration

In [ ]:
from pathlib import Path

RUN_MODE = "full"            # "smoke" (subject 1) or "full" (all 9)
SEED = 2026
np.random.seed(SEED); torch.manual_seed(SEED)

# MIRepNet released preprocessing: 8-30 Hz, 250 Hz, cue-locked 2-6 s (1000 samples),
# EA, then a 45-electrode channel template.
ELECTRODES = ["Fz","FC3","FC1","FCz","FC2","FC4","C5","C3","C1","Cz","C2","C4",
              "C6","CP3","CP1","CPz","CP2","CP4","P1","Pz","P2","POz"]
FS, BAND, FILTER_ORDER = 250, (8.0, 30.0), 5
WINDOW_S, N_OUT = (2.0, 6.0), 1000        # conv/pool geometry gives 61 tokens

MODES = ("native", "car", "median", "rest", "cz_ref", "lap_small", "lap_large")
ANCHOR = "native"            # MIRepNet re-references nothing; REVE's analogue is "car"
LAYERS = (0, 1, 2, 3, 4, 5, 6)            # patch embedding, then each transformer block
CONDS  = ("no_ea", "ea", "car_after", "car_ea")

# Probe identical to the REVE notebook: scaler -> PCA -> logistic, C by grouped CV.
PCA_COMPONENTS, C_GRID, CV_FOLDS = 128, (1e-4, 1e-3, 1e-2, 1e-1, 1.0), 5
RUN_BLOCK = 48               # IV-2a trials/run, used to form grouped-CV blocks

SUBJECTS = [1] if RUN_MODE == "smoke" else list(range(1, 10))
OUT = Path("/kaggle/working/mirepnet_consolidated"); OUT.mkdir(parents=True, exist_ok=True)
print("subjects:", SUBJECTS, "| modes:", MODES, "| anchor:", ANCHOR)

## 2. Preprocessing

`mirep_chain` reproduces MIRepNet's pipeline up to the alignment step; EA and the
channel template are applied afterwards so the four conditions can share one chain.

**Ordering.** The paper (Figs. 3-4) puts the channel template *before* EA. The
released code (`process_and_replace_loader`) runs `EA` first and interpolates
second. The code order is the only one that runs: the template lifts 22 real
channels to 45, so a covariance computed in template space has rank at most 22 out
of 45, and `fractional_matrix_power(R, -0.5)` on it is meaningless. Cell 2c
demonstrates this. We follow the code.

In [ ]:
from scipy.signal import butter, lfilter, resample as scipy_resample
from scipy.spatial.distance import cdist
from moabb.datasets import BNCI2014_001
from refshift.references import apply_reference, build_graph, _ea_fit, _ea_apply
from utils.channel_list import channel_positions, use_channels_names, BNCI2014001_chn_names

LABEL_MAP = {"left_hand":0,"right_hand":1,"feet":2,"tongue":3,
             "769":0,"770":1,"771":2,"772":3}
_B, _A = butter(FILTER_ORDER, [BAND[0]/(0.5*FS), BAND[1]/(0.5*FS)], btype="band")
_norm = lambda d: str(d).strip().lower().replace(" ", "_")


def load_long_trials(subject):
    """Cue-aligned 0-6 s trials in raw microvolts, read session-continuous."""
    data = BNCI2014_001().get_data(subjects=[subject])
    required = int(WINDOW_S[1] * FS)
    X, y, sessions = [], [], []
    for session_name, runs in data[subject].items():
        raws = []
        for raw in runs.values():
            raw = raw.copy().pick_channels(ELECTRODES, ordered=True)
            if float(raw.info["sfreq"]) != FS:
                raw.resample(FS)
            raws.append(raw)
        raw = mne.concatenate_raws(raws)
        assert raw.ch_names == ELECTRODES
        raw_uV = raw.get_data(units="uV")
        events, event_id = mne.events_from_annotations(raw, verbose=False)
        id_to_desc = {v: _norm(k) for k, v in event_id.items()}
        for sample, _, code_ in events:
            label = LABEL_MAP.get(id_to_desc.get(int(code_), ""))
            if label is None:
                continue
            seg = raw_uV[:, sample:sample + required]
            if seg.shape[-1] < required:      # only a session's final trial can be short
                continue
            X.append(seg); y.append(label); sessions.append(str(session_name))
    return np.stack(X).astype(np.float64), np.asarray(y, np.int64), np.asarray(sessions)

### 2b. MIRepNet's own chain: filter, window, template


In [ ]:
def mirep_chain(X_long, mode, graph, car_after=False):
    """Operator -> (optional CAR) -> 8-30 Hz -> 2-6 s -> 1000 samples. 22 channels."""
    X = apply_reference(X_long, mode, graph=graph)
    if car_after:
        X = apply_reference(X, "car", graph=graph)
    X = lfilter(_B, _A, X.astype(np.float64), axis=-1)
    X = X[..., int(WINDOW_S[0]*FS):int(WINDOW_S[1]*FS)]
    if X.shape[-1] != N_OUT:
        X = scipy_resample(X, N_OUT, axis=-1)
    return np.ascontiguousarray(X, dtype=np.float32)


def build_template(target, actual):
    """pad_missing_channels_diff, verbatim from MIRepNet utils/utils.py.
    Copied rather than imported: utils.utils and dataset.py import each other."""
    ex = np.array([channel_positions[c] for c in actual])
    tp = np.array([channel_positions[c] for c in target])
    W = np.zeros((len(target), len(actual)))
    for i, (tc, pos) in enumerate(zip(target, tp)):
        if tc in actual:
            W[i, actual.index(tc)] = 1.0
        else:
            d = cdist([pos], ex)[0]
            w = 1.0 / (d + 1e-6)
            W[i] = w / w.sum()
    return W


CH_UPPER = [c.upper() for c in ELECTRODES]
assert CH_UPPER == BNCI2014001_chn_names, "channel order differs from MIRepNet's list"
W_TEMPLATE = build_template(use_channels_names, CH_UPPER)
GRAPH = build_graph(ELECTRODES, include_rest=("rest" in MODES))

to_template = lambda X: np.ascontiguousarray(
    np.einsum("ij,njt->nit", W_TEMPLATE, X.astype(np.float64)), dtype=np.float32)

def zscore(X):                                # scale only; see the ablation note below
    return (X - X.mean(-1, keepdims=True)) / (X.std(-1, keepdims=True) + 1e-8)

def session_split(sessions):
    train = sorted(set(map(str, sessions)))[0]      # "0train" sorts before "1test"
    tr = np.flatnonzero(sessions == train)
    te = np.flatnonzero(sessions != train)
    return tr, te, np.arange(len(tr)) // RUN_BLOCK

In [ ]:
_X, _y, _s = load_long_trials(1)
for name in np.unique(_s):
    print(f"session {name}: {int((_s == name).sum())} trials")
print("per-trial shape:", _X.shape[1:], "| classes:", np.unique(_y))
print(f"template: {W_TEMPLATE.shape[1]} -> {W_TEMPLATE.shape[0]} channels")
del _X, _y, _s

### 2c. The channel template cannot canonicalise a reference

The interpolation weights are normalised, so `W` has rows summing to 1 and
`W @ 1 = 1`. Any reference of the form `x - 1 f(x)` (CAR, median, cz_ref, REST)
therefore passes through the template unchanged. Whatever reference robustness
MIRepNet has must come from EA, not from the template. Data-free.

In [ ]:
print(f"rows sum to 1     : {np.allclose(W_TEMPLATE.sum(1), 1.0)}")
print(f"||W @ 1 - 1||     : {np.abs(W_TEMPLATE @ np.ones(len(CH_UPPER)) - 1).max():.2e}")
print(f"dropped from IV-2a: {[c for c in CH_UPPER if c not in use_channels_names]}")
print(f"interpolated      : {sum(1 for c in use_channels_names if c not in CH_UPPER)} of 45")

### 2d. Why we follow the code order, and where the released EA fails

Two checks on MIRepNet's alignment step. First, EA in template space is
rank-deficient by construction. Second, the released `EA()` inverts the covariance
with no rank handling, so it breaks on references whose covariance is singular.
`_ea_fit` truncates eigen-directions below `rel_floor * lambda_max` instead.

In [ ]:
from scipy.linalg import fractional_matrix_power

def EA_released(x):                           # verbatim from MIRepNet utils/utils.py
    cov = np.stack([np.cov(t) for t in x])
    S = fractional_matrix_power(cov.mean(0), -0.5)
    return np.stack([S @ t for t in x])

_X, _y, _s = load_long_trials(1)
_tr, _te, _ = session_split(_s)
_probe = mirep_chain(_X, "native", GRAPH)[_tr][:64].astype(np.float64)

Rt = np.stack([np.cov(t) for t in to_template(_probe).astype(np.float64)]).mean(0)
print("paper order (template -> EA): covariance is "
      f"{Rt.shape[0]}x{Rt.shape[0]}, rank {np.linalg.matrix_rank(Rt)}, "
      f"cond {np.linalg.cond(Rt):.1e}  -> not invertible; we use the code order\n")

rows = []
for m in MODES:
    Z = mirep_chain(_X, m, GRAPH)[_tr][:64].astype(np.float64)
    R = np.stack([np.cov(t) for t in Z]).mean(0)
    rel = EA_released(Z)
    ours = _ea_apply(Z.astype(np.float32), _ea_fit(Z.astype(np.float32)))
    rows.append({"reference": m, "rank(R)": int(np.linalg.matrix_rank(R)),
                 "cond(R)": f"{np.linalg.cond(R):.1e}",
                 "released_finite": bool(np.isfinite(rel).all()),
                 "released_max": f"{np.abs(rel).max():.2e}",
                 "ranksafe_max": f"{np.abs(ours).max():.2e}"})
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nfull rank would be {len(ELECTRODES)}")
del _X, _y, _s, _probe; gc.collect();

## 3. Features and probe

The probe is identical to the REVE notebook so the two are read the same way:
scaler, PCA, logistic regression, with `C` chosen by grouped CV over run blocks of
the train session. Metric is balanced accuracy.

MIRepNet's encoder in eval mode returns `(pooled, logits)`; `pooled` is the mean
over its 61 temporal-spatial tokens, a 256-d vector. `pooling="mean"` is the model's
own aggregation; the alternatives exist only for the robustness grid in Experiment 5.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.metrics import balanced_accuracy_score

def _aggregate(tokens, pooling):
    if pooling == "mean":    return tokens.mean(dim=1)                       # the model's own
    if pooling == "meanstd": return torch.cat([tokens.mean(1), tokens.std(1)], -1)
    if pooling == "no":      return tokens.flatten(1)
    raise ValueError(pooling)

@torch.inference_mode()
def mirep_features(X45, layer=None, pooling="mean", batch=32):
    """X45 is (N, 45, 1000). layer=None uses the full encoder."""
    out = []
    for i in range(0, len(X45), batch):
        xb = torch.from_numpy(X45[i:i+batch]).float().to(DEVICE)
        h = mirep.embedding(xb)
        depth = len(mirep.transformer) if layer is None else layer
        for block in list(mirep.transformer)[:depth]:
            h = block(h)
        out.append(_aggregate(h, pooling).float().cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def fit_probe(X, y, groups, *, pca=PCA_COMPONENTS, C=None):
    n_comp = min(pca, X.shape[1], len(X) - 1) if pca else None
    steps = [("scale", StandardScaler())]
    if n_comp:
        steps.append(("pca", PCA(n_components=n_comp, svd_solver="randomized",
                                 random_state=SEED)))
    steps.append(("clf", LogisticRegression(solver="lbfgs", max_iter=5000,
                                            random_state=SEED)))
    pipe = Pipeline(steps)
    if C is not None:
        return pipe.set_params(clf__C=C).fit(X, y)
    cv = StratifiedGroupKFold(n_splits=min(CV_FOLDS, len(np.unique(groups))),
                              shuffle=True, random_state=SEED)
    search = GridSearchCV(pipe, {"clf__C": list(C_GRID)}, scoring="balanced_accuracy",
                          cv=cv, refit=True)
    search.fit(X, y, groups=groups)
    return search.best_estimator_

def bacc(probe, X, y):
    return float(balanced_accuracy_score(y, probe.predict(X)))

## 4. Per-subject feature store

Four conditions. `no_ea` and `car_after` match the REVE notebook's definitions
exactly (no alignment), which is what makes the two sets of numbers commensurable.
`car_ea` is an addition: CAR-after gives every operator the same null space, so EA
truncates the same direction for all of them, which turns out to matter.

`no_ea` and `car_after` z-score for scale rather than passing raw microvolts.
MIRepNet always runs EA, so `embedding.bn` carries running statistics estimated on
whitened input; unnormalised data would be out of distribution for reasons that have
nothing to do with references. Both are off-pipeline ablations either way.

In [ ]:
def ea_input(X, tr, te):
    """Euclidean-align train and test separately: no whitener crosses the split."""
    Z = np.empty_like(X)
    Z[tr] = _ea_apply(X[tr], _ea_fit(X[tr]))
    Z[te] = _ea_apply(X[te], _ea_fit(X[te]))
    return Z


def condition_input(X_long, mode, cond, tr, te):
    """The four conditions, defined in exactly one place.

        no_ea      operator            -> z-score      (template alone)
        ea         operator            -> EA           (MIRepNet as released)
        car_after  operator -> CAR     -> z-score      (matches the REVE notebook)
        car_ea     operator -> CAR     -> EA           (CAR-after plus alignment)

    Returns (N, 45, 1000), ready for the encoder.
    """
    X = mirep_chain(X_long, mode, GRAPH, car_after=cond in ("car_after", "car_ea"))
    X = ea_input(X, tr, te) if cond in ("ea", "car_ea") else zscore(X)
    return to_template(X)


def build_store(subject):
    X_long, y, sessions = load_long_trials(subject)
    tr, te, groups = session_split(sessions)
    feat = {c: {m: mirep_features(condition_input(X_long, m, c, tr, te))
                for m in MODES} for c in CONDS}
    return dict(subject=subject, y=y, tr=tr, te=te, groups=groups, feat=feat)

## Experiment 1 - Reference-mismatch transfer matrix

In [ ]:
def experiment_matrix(store):
    y, tr, te, g = store["y"], store["tr"], store["te"], store["groups"]
    rows = []
    for cond in CONDS:
        f = store["feat"][cond]
        probes = {m: fit_probe(f[m][tr], y[tr], g) for m in MODES}
        for a in MODES:
            for b in MODES:
                rows.append(dict(subject=store["subject"], condition=cond,
                                 train_ref=a, test_ref=b,
                                 bacc=bacc(probes[a], f[b][te], y[te])))
    return rows

## 5. Run experiment 1 (one pass per subject)

In [ ]:
matrix_rows = []
for s in SUBJECTS:
    print(f"subject {s} ...", flush=True)
    store = build_store(s)
    matrix_rows += experiment_matrix(store)
    del store; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

matrix = pd.DataFrame(matrix_rows); matrix.to_csv(OUT/"matrix.csv", index=False)
print("saved matrix")

### Experiment 1 result

In [ ]:
def gap(df):
    diag = df[df.train_ref == df.test_ref].bacc.mean()
    off  = df[df.train_ref != df.test_ref].bacc.mean()
    return diag, off, diag - off

def matrix_of(cond):
    return (matrix[matrix.condition == cond]
            .groupby(["train_ref", "test_ref"]).bacc.mean()
            .unstack("test_ref").reindex(index=MODES, columns=MODES) * 100)

for cond in CONDS:
    d, o, gp = gap(matrix[matrix.condition == cond])
    print(f"{cond:9s} matched={100*d:5.1f}%  mismatched={100*o:5.1f}%  gap={100*gp:5.2f}pp")

for cond in CONDS:
    print(f"\n[{cond}] matrix (rows=train, cols=test), %:")
    print(matrix_of(cond).round(1).to_string())

# CAR-after collapses every global reference onto CAR, so its global block is
# degenerate and its off-diagonal gap is deflated. The cells that test the
# hypothesis are global<->spatial.
spatial = ["lap_small", "lap_large"]
glob = [m for m in MODES if m not in spatial]
print("\nGlobal<->spatial cross-family transfer, balanced accuracy %:")
for cond in CONDS:
    s = matrix[matrix.condition == cond]
    g2s = s[s.train_ref.isin(glob) & s.test_ref.isin(spatial)].bacc.mean() * 100
    s2g = s[s.train_ref.isin(spatial) & s.test_ref.isin(glob)].bacc.mean() * 100
    print(f"  {cond:9s} global->spatial={g2s:5.1f}  spatial->global={s2g:5.1f}")

def gap_ci(df, n_boot=10000, seed=SEED):
    per = df.groupby("subject").apply(
        lambda d: d[d.train_ref == d.test_ref].bacc.mean()
                - d[d.train_ref != d.test_ref].bacc.mean())
    v = per.to_numpy(float)
    rng = np.random.default_rng(seed)
    boot = rng.choice(v, (n_boot, v.size), replace=True).mean(1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return v.mean() * 100, lo * 100, hi * 100, v.size

print("\nSubject-level gap [95% CI]:")
for cond in CONDS:
    m, lo, hi, n = gap_ci(matrix[matrix.condition == cond])
    print(f"  {cond:9s} {m:5.2f}pp  [{lo:5.2f}, {hi:5.2f}]  n={n}")

### Where EA fails

EA reduces any invertible **linear** operator to a rotation, so an operator it
cannot repair is worth naming. `median` is the only non-linear one, and the
only operator missing from the invertibility table above.


In [ ]:
# Per-test-reference view: EA repairs linear operators up to a rotation, so any
# operator it fails on is worth naming. median is the only non-linear one.
print("\n[ea] per-test-reference gap, %:")
M = matrix_of("ea").to_numpy(float)
for j, t in enumerate(MODES):
    off = np.delete(M[:, j], j)
    print(f"  test on {t:10s} matched={M[j,j]:5.1f}  mismatched={off.mean():5.1f}  "
          f"gap={M[j,j]-off.mean():5.1f}")

## Operator invertibility (why global collapses and spatial does not)

Identical table to the REVE notebook. `median` is absent because it is non-linear
and has no fixed matrix, which is also why EA has no repair guarantee for it.

In [ ]:
from refshift import contrast_recovery_report
print(contrast_recovery_report(ELECTRODES, modes=MODES).to_string(index=False))

## Experiment 1b - Online few-trial target EA (the deployable test)

How many target-session trials does the EA whitener need before the mismatch is
gone? Probe is trained on the anchor reference; the evaluation block is held out of
the whitener fit at every `k` so the numbers are commensurable.

In [ ]:
K_GRID = (1, 2, 4, 8, 16, 32)
KMAX = max(K_GRID)

def whiten_prefix(target, k):
    return _ea_apply(target[KMAX:], _ea_fit(target[:k]))

def whiten_full(target):
    return _ea_apply(target[KMAX:], _ea_fit(target))

kill_rows = []
for s in SUBJECTS:
    X_long, y, sessions = load_long_trials(s)
    tr, te, g = session_split(sessions)
    y_eval = y[te][KMAX:]
    anchor_in = mirep_chain(X_long, ANCHOR, GRAPH)
    probe = fit_probe(mirep_features(to_template(
        _ea_apply(anchor_in[tr], _ea_fit(anchor_in[tr])))), y[tr], g)
    for test_ref in MODES:
        target = mirep_chain(X_long, test_ref, GRAPH)[te]
        for k in K_GRID:
            feat = mirep_features(to_template(whiten_prefix(target, k)))
            kill_rows.append(dict(subject=s, test_ref=test_ref, k=str(k),
                                  matched=(test_ref == ANCHOR),
                                  bacc=bacc(probe, feat, y_eval)))
        feat = mirep_features(to_template(whiten_full(target)))
        kill_rows.append(dict(subject=s, test_ref=test_ref, k="full",
                              matched=(test_ref == ANCHOR),
                              bacc=bacc(probe, feat, y_eval)))
    del X_long; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

kill = pd.DataFrame(kill_rows); kill.to_csv(OUT/"online_ea.csv", index=False)
order = [str(k) for k in K_GRID] + ["full"]
piv = kill.groupby(["k", "matched"]).bacc.mean().unstack("matched") * 100
piv.columns = ["matched" if c else "mismatched" for c in piv.columns]
piv["gap"] = piv["matched"] - piv["mismatched"]
print(f"Target EA by calibration trials k ({ANCHOR} probe; k='full' = full-session target EA):")
print(piv.reindex(order).round(1).to_string())

## Experiment 2 - Depth trajectory

Anchor-based, exactly as in the REVE notebook: one probe trained on `ANCHOR`,
matched is that probe on the anchor reference, mismatched is its mean over the other
six. This is a different estimator from the Experiment 1 matrix gap, which averages
over all seven anchors, so the two numbers are not expected to agree.

`rel_gap` divides the gap by accuracy above chance. A probe near chance cannot show
a gap, so the raw gap tracks accuracy for trivial reasons.

In [ ]:
traj_rows = []
for s in SUBJECTS:
    X_long, y, sessions = load_long_trials(s)
    tr, te, g = session_split(sessions)
    # REVE runs these on unaligned input; EA would close the gap being measured.
    inputs = {m: condition_input(X_long, m, "no_ea", tr, te) for m in MODES}
    for layer in LAYERS:
        feat = {m: mirep_features(inputs[m], layer=layer) for m in MODES}
        probe = fit_probe(feat[ANCHOR][tr], y[tr], g)
        matched = bacc(probe, feat[ANCHOR][te], y[te])
        mism = np.mean([bacc(probe, feat[m][te], y[te]) for m in MODES if m != ANCHOR])
        traj_rows.append(dict(subject=s, layer=layer, matched=matched, mismatched=mism))
    del inputs; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

traj = pd.DataFrame(traj_rows); traj.to_csv(OUT/"trajectory.csv", index=False)
CHANCE = 100.0 / 4          # IV-2a is 4-class
t = traj.groupby("layer")[["matched", "mismatched"]].mean() * 100
t["gap"] = t.matched - t.mismatched
t["above_chance"] = t.matched - CHANCE
t["rel_gap"] = (t.gap / t.above_chance).round(2)
print("Mean over subjects by layer (%):")
print(t.round(1).to_string())

## Experiment 5 - Robustness to representation choice

Does the gap survive a different aggregation or a different PCA width? MIRepNet has
no context token, so the pooling variants are its own mean, mean concatenated with
std, and the full flattened token sequence.

In [ ]:
def anchored_gap(feat, y, tr, te, g, pca):
    proj = Pipeline([("scale", StandardScaler()),
                     ("pca", PCA(n_components=min(pca, len(tr) - 1),
                                 svd_solver="randomized",
                                 random_state=SEED))]).fit(feat[ANCHOR][tr])
    Z = lambda m, idx: proj.transform(feat[m][idx])
    clf = fit_probe(Z(ANCHOR, tr), y[tr], g, pca=None)
    matched = bacc(clf, Z(ANCHOR, te), y[te])
    mism = np.mean([bacc(clf, Z(m, te), y[te]) for m in MODES if m != ANCHOR])
    return matched - mism

rob_rows = []
for s in SUBJECTS:
    X_long, y, sessions = load_long_trials(s)
    tr, te, g = session_split(sessions)
    # REVE runs these on unaligned input; EA would close the gap being measured.
    inputs = {m: condition_input(X_long, m, "no_ea", tr, te) for m in MODES}
    for pooling in ("mean", "meanstd", "no"):
        feat = {m: mirep_features(inputs[m], pooling=pooling) for m in MODES}
        for pca in (64, 128, 256):
            rob_rows.append(dict(subject=s, pooling=pooling, pca=pca,
                                 gap=anchored_gap(feat, y, tr, te, g, pca)))
    del inputs; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

rob = pd.DataFrame(rob_rows); rob.to_csv(OUT/"robustness.csv", index=False)
cfg = rob.groupby(["pooling", "pca"]).gap.mean() * 100
print("Per-configuration mean matched-minus-mismatched gap (pp):")
print(cfg.unstack("pca").round(1).to_string())
print(f"\nConfig-mean gap range: {cfg.min():.1f} to {cfg.max():.1f} pp across all "
      f"{len(cfg)} pooling x PCA configurations.")

## 6. Outcome summary

In [ ]:
print("="*64)
print("MIRepNet reference-mismatch: outcome summary")
print("="*64)
d0, o0, g0 = gap(matrix[matrix.condition == "no_ea"])
d1, o1, g1 = gap(matrix[matrix.condition == "ea"])
_, _, g2 = gap(matrix[matrix.condition == "car_ea"])
print(f"1.  Gap no-adapt : {100*g0:5.2f}pp (matched {100*d0:.1f} / mismatched {100*o0:.1f})")
print(f"    Gap full EA  : {100*g1:5.2f}pp  -> EA removes {100*(g0-g1):.1f}pp")
print(f"    Gap CAR+EA   : {100*g2:5.2f}pp  -> CAR-after removes a further {100*(g1-g2):.1f}pp")

_sp = ["lap_small", "lap_large"]; _gl = [m for m in MODES if m not in _sp]
def _xfam(cond):
    s = matrix[matrix.condition == cond]
    return (s[s.train_ref.isin(_gl) & s.test_ref.isin(_sp)].bacc.mean() * 100,
            s[s.train_ref.isin(_sp) & s.test_ref.isin(_gl)].bacc.mean() * 100)
_rw, _ca = _xfam("no_ea"), _xfam("car_after")
print(f"1c. CAR-after global<->spatial: g->s {_rw[0]:.1f}->{_ca[0]:.1f}  "
      f"s->g {_rw[1]:.1f}->{_ca[1]:.1f}  (global refs collapse to CAR; Laplacians do not)")

kg = kill.groupby("k").apply(
    lambda d: (d[d.matched].bacc.mean() - d[~d.matched].bacc.mean()) * 100)
print(f"1b. Online target EA gap: k=1 {kg.get('1', float('nan')):.1f}pp -> "
      f"k=32 {kg.get('32', float('nan')):.1f}pp -> full {kg.get('full', float('nan')):.1f}pp")

tg = traj.groupby("layer")[["matched", "mismatched"]].mean()
print(f"2.  Depth gap    : {100*(tg.matched-tg.mismatched).iloc[0]:.1f}pp (layer {LAYERS[0]}) -> "
      f"{100*(tg.matched-tg.mismatched).iloc[-1]:.1f}pp (layer {LAYERS[-1]})")

cfg = rob.groupby(["pooling", "pca"]).gap.mean() * 100
print(f"3.  Pooling/PCA robustness: config-mean gap {cfg.min():.1f}-{cfg.max():.1f}pp "
      f"across {len(cfg)} configs")
print("\nNote: 1/1c use the full 7x7 matrix; 2/3 are anchor-based single-probe gaps.")
print(f"      Anchor here is {ANCHOR!r} (MIRepNet re-references nothing); REVE uses 'car'.")